# Pretrained Agents: Load, Evaluate, and Compare

Load pretrained models from HuggingFace and benchmark them against baselines.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/christianWissmann85/essence-wars/blob/master/notebooks/06_pretrained_agents.ipynb)

In [ ]:
# Colab setup - run this cell if using Google Colab
# import sys
#
# if 'google.colab' in sys.modules:
#     print("Installing essence-wars from PyPI...")
#     !pip install essence-wars[train]
#
#     print("\nCloning repository for data files...")
#     !git clone --depth 1 https://github.com/christianWissmann85/essence-wars.git
#     import os
#     os.chdir('essence-wars')
#
#     print("\nInstallation complete!")

In [ ]:
# Setup: Change to repo root directory (required for data files)
import os
from pathlib import Path


def find_repo_root():
    path = Path.cwd()
    while path != path.parent:
        if (path / 'data' / 'cards').exists():
            return path
        path = path.parent
    return None

repo_root = find_repo_root()
if repo_root:
    os.chdir(repo_root)
    print(f"Working directory: {os.getcwd()}")
else:
    print("Warning: Could not find repo root.")

## 1. Available Pretrained Models

We have several pretrained PPO agents available on HuggingFace:

| Model | Win Rate vs Greedy | Architecture | Specialty |
|-------|-------------------|--------------|----------|
| **PPO-Argentum** | 72% | Embedded | Argentum faction |
| **PPO-Flat** | 71% | Flat | Generalist |
| **PPO-Embedded** | 65% | Embedded | Generalist |
| **PPO-Symbiote** | 65% | Embedded | Symbiote faction |
| **PPO-Obsidion** | 62% | Embedded | Obsidion faction |

View all models: [huggingface.co/Chris-Essence-Wars](https://huggingface.co/Chris-Essence-Wars)

## 2. Load a Pretrained Model

In [ ]:
from huggingface_hub import hf_hub_download

# Download the best model (PPO-Argentum, 72% vs Greedy)
model_path = hf_hub_download(
    repo_id="Chris-Essence-Wars/ppo-argentum",
    filename="model.pt"
)
print(f"Downloaded model to: {model_path}")

In [ ]:
from essence_wars.benchmark import NeuralAgent

# Load as a NeuralAgent (ready for evaluation)
agent = NeuralAgent.from_checkpoint(model_path, name="PPO-Argentum")
print(f"Loaded agent: {agent.name}")

## 3. Watch the Agent Play

In [ ]:
import numpy as np
from essence_wars._core import PyGame
from essence_wars.viz import GameRenderer

# Create a game
game = PyGame(deck1="architect_fortify", deck2="broodmother_swarm")
game.reset(seed=42)

renderer = GameRenderer()
print("Initial state:")
renderer.print(game)

In [ ]:
# Play a game: PPO-Argentum vs GreedyBot
game.reset(seed=123)
agent.reset()

actions_taken = 0
while not game.is_done():
    current_player = game.current_player()
    obs = np.array(game.observe(), dtype=np.float32)
    mask = np.array(game.action_mask(), dtype=np.float32)
    
    if current_player == 0:
        # Our pretrained agent
        action = agent.select_action(obs, mask)
    else:
        # Greedy opponent
        action = game.greedy_action()
    
    game.step(action)
    actions_taken += 1
    
    if actions_taken > 200:
        print("Game too long, breaking...")
        break

# Check result
reward = game.get_reward(0)
result = "Win" if reward > 0 else ("Loss" if reward < 0 else "Draw")
print(f"\nGame finished in {actions_taken} actions")
print(f"Result: {result}")
print(f"\nFinal state:")
renderer.print(game)

## 4. Benchmark Evaluation

Run a proper evaluation against standard baselines.

In [ ]:
from essence_wars.benchmark import EssenceWarsBenchmark

# Quick evaluation (40 games)
benchmark = EssenceWarsBenchmark(verbose=True)
results = benchmark.quick_evaluate(agent, games=40)

print(f"\n=== Quick Evaluation Results ===")
print(f"Win rate vs Random: {results['vs_random']:.1%}")
print(f"Win rate vs Greedy: {results['vs_greedy']:.1%}")
print(f"Estimated Elo: {results['elo']:.0f}")

In [ ]:
# Full evaluation (100 games per opponent) - takes longer but more accurate
# Uncomment to run:

# full_results = benchmark.evaluate(
#     agent,
#     games_per_opponent=100,
#     baselines=["random", "greedy", "mcts50", "mcts100"]
# )
# print(full_results.summary())

## 5. Compare Multiple Models

Download and compare several pretrained agents.

In [ ]:
# Download multiple models
models_to_compare = [
    ("Chris-Essence-Wars/ppo-argentum", "PPO-Argentum"),
    ("Chris-Essence-Wars/ppo-flat", "PPO-Flat"),
    ("Chris-Essence-Wars/ppo-embedded", "PPO-Embedded"),
]

agents = []
for repo_id, name in models_to_compare:
    try:
        path = hf_hub_download(repo_id=repo_id, filename="model.pt")
        agent = NeuralAgent.from_checkpoint(path, name=name)
        agents.append(agent)
        print(f"Loaded: {name}")
    except Exception as e:
        print(f"Could not load {name}: {e}")

print(f"\nLoaded {len(agents)} agents for comparison")

In [ ]:
# Evaluate all agents
comparison_results = []

for agent in agents:
    print(f"\nEvaluating {agent.name}...")
    results = benchmark.quick_evaluate(agent, games=40)
    comparison_results.append({
        'name': agent.name,
        'vs_random': results['vs_random'],
        'vs_greedy': results['vs_greedy'],
        'elo': results['elo'],
    })

In [ ]:
import pandas as pd

# Display comparison table
df = pd.DataFrame(comparison_results)
df = df.sort_values('elo', ascending=False)
df['vs_random'] = df['vs_random'].apply(lambda x: f"{x:.1%}")
df['vs_greedy'] = df['vs_greedy'].apply(lambda x: f"{x:.1%}")
df['elo'] = df['elo'].apply(lambda x: f"{x:.0f}")
df.columns = ['Agent', 'vs Random', 'vs Greedy', 'Elo']
print("\n=== Agent Comparison ===")
print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = [r['name'] for r in comparison_results]
vs_greedy = [r['vs_greedy'] * 100 for r in comparison_results]
elos = [r['elo'] for r in comparison_results]

# Win rate chart
colors = plt.cm.viridis([0.2, 0.5, 0.8])[:len(names)]
axes[0].barh(names, vs_greedy, color=colors)
axes[0].set_xlabel('Win Rate vs Greedy (%)')
axes[0].set_title('Performance vs GreedyBot')
axes[0].axvline(50, color='red', linestyle='--', alpha=0.5, label='50% baseline')
for i, v in enumerate(vs_greedy):
    axes[0].text(v + 1, i, f'{v:.1f}%', va='center')

# Elo chart  
axes[1].barh(names, elos, color=colors)
axes[1].set_xlabel('Elo Rating')
axes[1].set_title('Estimated Elo Ratings')
axes[1].axvline(1300, color='red', linestyle='--', alpha=0.5, label='GreedyBot (1300)')
for i, v in enumerate(elos):
    axes[1].text(v + 10, i, f'{v:.0f}', va='center')

plt.tight_layout()
plt.show()

## 6. Head-to-Head Comparison

Play agents directly against each other.

In [ ]:
def head_to_head(agent1, agent2, num_games=20):
    """Play two agents against each other."""
    wins = [0, 0]
    draws = 0
    
    for game_idx in range(num_games):
        game = PyGame()
        game.reset(seed=game_idx + 5000)
        agent1.reset()
        agent2.reset()
        
        # Alternate who goes first
        agents = [agent1, agent2] if game_idx % 2 == 0 else [agent2, agent1]
        
        while not game.is_done():
            current_player = game.current_player()
            obs = np.array(game.observe(), dtype=np.float32)
            mask = np.array(game.action_mask(), dtype=np.float32)
            
            action = agents[current_player].select_action(obs, mask)
            game.step(action)
        
        reward = game.get_reward(0)
        if game_idx % 2 == 0:  # agent1 was player 0
            if reward > 0:
                wins[0] += 1
            elif reward < 0:
                wins[1] += 1
            else:
                draws += 1
        else:  # agent2 was player 0
            if reward > 0:
                wins[1] += 1
            elif reward < 0:
                wins[0] += 1
            else:
                draws += 1
    
    return wins, draws

if len(agents) >= 2:
    print(f"\n=== Head-to-Head: {agents[0].name} vs {agents[1].name} ===")
    wins, draws = head_to_head(agents[0], agents[1], num_games=20)
    print(f"{agents[0].name}: {wins[0]} wins")
    print(f"{agents[1].name}: {wins[1]} wins")
    print(f"Draws: {draws}")

## 7. Submit Your Own Agent

Think you can beat these models? Submit your agent to the leaderboard!

In [ ]:
# Submission example (run from command line)
print("""
To submit your trained agent:

1. Train your model (see 04_behavioral_cloning.ipynb or 05_alphazero_training.ipynb)

2. Run the submission script:
   python python/scripts/submit_agent.py \\
       --checkpoint your_model.pt \\
       --name "Your Agent Name" \\
       --repo-id yourusername/essence-wars-agent

3. View the leaderboard:
   https://huggingface.co/spaces/Chris-Essence-Wars/essence-wars-leaderboard

See the full guide:
   https://github.com/christianWissmann85/essence-wars/blob/master/docs/SUBMIT_AGENT.md
""")

## Resources

- [Leaderboard](https://huggingface.co/spaces/Chris-Essence-Wars/essence-wars-leaderboard) - See all ranked agents
- [Pretrained Models](https://huggingface.co/Chris-Essence-Wars) - Download more models
- [Submission Guide](https://github.com/christianWissmann85/essence-wars/blob/master/docs/SUBMIT_AGENT.md) - Submit your agent
- [Benchmark Methodology](https://github.com/christianWissmann85/essence-wars/blob/master/docs/benchmark-methodology.md) - How we evaluate
- [GitHub Repository](https://github.com/christianWissmann85/essence-wars) - Full source code